In [1]:
import torch
from torch import nn

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
class SingleHeadAttention(nn.Module):
    def __init__(self, model_dim: int, head_size: int, mask: bool = True):
        super().__init__()
        self.key_layer = nn.Linear(model_dim, head_size, bias=False)
        self.query_layer = nn.Linear(model_dim, head_size, bias=False)
        self.value_layer = nn.Linear(model_dim, head_size, bias=False)
        self.mask = mask

    def forward(self, query, key=None, value=None):
        """
        query: (B, T_q, D)
        key:   (B, T_k, D) or None (defaults to query)
        value: (B, T_v, D) or None (defaults to key)
        """
        if key is None:
            key = query
        if value is None:
            value = key
        
        k = self.key_layer(key)
        q = self.query_layer(query)
        v = self.value_layer(value)

        scores = q @ torch.transpose(k, 1, 2)
        context_length, attention_dim = k.shape[1], k.shape[2]
        scores = scores / (attention_dim ** 0.5)

        if self.mask:
            lower_triangular = torch.tril(torch.ones(context_length, context_length))
            mask = (lower_triangular == 0).to(device)
            scores = scores.masked_fill(mask, float('-inf'))

        scores = nn.functional.softmax(scores, dim = -1)

        return scores @ v

In [4]:
embedding_dim = 2
attention_dim = 3

attention = SingleHeadAttention(embedding_dim, attention_dim, mask=True)
attention = attention.to(device)

embedded = [
    [[-1.4381, 0.1232],
    [-0.1080, 0.3458]],
    [[0.1929, -0.8567],
    [-0.1160, 1.2547]]
]

embedded = torch.tensor(embedded, dtype=torch.float32).to(device)
output = attention(embedded)
print("Output shape:", output.shape)
print("Output:", output)

Output shape: torch.Size([2, 2, 3])
Output: tensor([[[-0.1550, -0.5274, -0.5690],
         [-0.1620, -0.3901, -0.3911]],

        [[ 0.4130,  0.6127,  0.5146],
         [-0.1052, -0.1380, -0.1082]]], device='cuda:0',
       grad_fn=<UnsafeViewBackward0>)
